# 01. YOLO11 영상 inference 시각화

이 노트북은 YOLO11 weight로 영상 파일 또는 영상 폴더를 inference하고, bbox와 `class score` 라벨이 그려진 MP4 영상을 저장합니다.

기본값은 안전하게 실행되지 않도록 되어 있습니다.

- `RUN_PREVIEW = False`: 영상 목록과 metadata만 확인하는 preview도 기본 비활성화
- `RUN_INFERENCE = False`: 실제 model load/inference/video write 비활성화
- `DRY_RUN = True`: inference cell을 실행해도 preview만 수행

실제 실행 순서:

1. `INPUT_PATH`, `OUT_DIR`, `MODEL_WEIGHTS`, `CLASS_YAML`을 실제 경로로 수정합니다.
2. class check cell에서 28개 class 순서가 맞는지 확인합니다.
3. 먼저 `RUN_PREVIEW=True`로 영상 목록과 metadata를 확인합니다.
4. 문제가 없으면 `RUN_INFERENCE=True`, `DRY_RUN=False`로 바꿔 실행합니다.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").is_file() and (path / "src").is_dir():
            return path
    raise RuntimeError("repo root를 찾지 못했습니다. notebook을 repo 안에서 실행하거나 REPO_ROOT를 직접 지정하세요.")

REPO_ROOT = find_repo_root(Path.cwd())
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("REPO_ROOT:", REPO_ROOT)

from labelstudio_bbox_tools.video_inference.classes import load_class_names, make_class_color_map
from labelstudio_bbox_tools.video_inference.yolo11 import run_yolo11_video_inference


In [ ]:
# ===== 사용자가 주로 바꾸는 설정 =====

INPUT_PATH = Path("/path/to/video_or_video_folder")
OUT_DIR = Path("/path/to/video_inference_outputs")
MODEL_WEIGHTS = Path("/path/to/yolo11_best.pt")
CLASS_YAML = Path("/path/to/data.yaml")
MANUAL_CLASSES = []  # CLASS_YAML을 쓰면 비워둡니다. yaml 없이 직접 넣을 때만 사용하세요.

DEVICE = "0"          # GPU 0번 사용 예시. CPU면 None 또는 "cpu"로 둡니다.
IMGSZ = 1280           # None, 640, 1280, 또는 (width, height)
CONF = 0.25
IOU = 0.45

# 폴더 입력일 때 기본은 대상 폴더 내부만 탐색합니다. 하위 폴더까지 보려면 True.
RECURSIVE = False
MAX_VIDEOS = 1         # 처음엔 1개만 확인 추천. 전체 실행 시 None.
FRAME_STRIDE = 1       # 1이면 모든 frame 저장. 2면 2 frame마다 1개씩 저장.
START_SECONDS = None   # 예: 10.0
END_SECONDS = None     # 예: 60.0
MAX_FRAMES = 300       # 처음엔 짧게 확인 추천. 전체 실행 시 None.

EXPECTED_CLASS_COUNT = 28
STRICT_CLASS_COUNT = True

FONT_PATH = None       # None이면 Nanum/Noto CJK font를 자동 탐색합니다.
FONT_SIZE = 20
LINE_WIDTH = 3
RUN_NAME = None        # None이면 시간 기반 폴더명 자동 생성.
OVERWRITE = False

# 안전 플래그
RUN_PREVIEW = False
RUN_INFERENCE = False
DRY_RUN = True


In [ ]:
# class YAML과 color map 확인 cell입니다.
# class id 순서가 학습 때 사용한 28개 class 순서와 반드시 같아야 합니다.

if CLASS_YAML.exists() or MANUAL_CLASSES:
    class_names = load_class_names(
        class_yaml=CLASS_YAML if CLASS_YAML.exists() else None,
        manual_classes=MANUAL_CLASSES or None,
        expected_count=EXPECTED_CLASS_COUNT,
        strict_count=STRICT_CLASS_COUNT,
    )
    color_map = make_class_color_map(class_names)
    print("class_count:", len(class_names))
    for idx, name in enumerate(class_names):
        print(f"{idx:02d}: {name:24s} color={color_map[name]}")
else:
    print("CLASS_YAML 경로를 실제 data.yaml로 수정하거나 MANUAL_CLASSES를 채워주세요.")


In [ ]:
# 영상 목록과 metadata만 확인합니다. model은 load하지 않습니다.

if not RUN_PREVIEW:
    print("RUN_PREVIEW=False 이므로 preview를 실행하지 않습니다.")
else:
    preview = run_yolo11_video_inference(
        input_path=INPUT_PATH,
        out_dir=OUT_DIR,
        model_weights=MODEL_WEIGHTS,
        class_yaml=CLASS_YAML if CLASS_YAML.exists() else None,
        manual_classes=MANUAL_CLASSES or None,
        recursive=RECURSIVE,
        max_videos=MAX_VIDEOS,
        expected_class_count=EXPECTED_CLASS_COUNT,
        strict_class_count=STRICT_CLASS_COUNT,
        run_name=RUN_NAME,
        dry_run=True,
    )
    print(preview.as_dict())


In [ ]:
# 실제 YOLO11 inference와 시각화 영상 저장 cell입니다.
# 실행 전 RUN_INFERENCE=True, DRY_RUN=False로 바꿔야 실제 영상이 저장됩니다.

if not RUN_INFERENCE:
    print("RUN_INFERENCE=False 이므로 실제 inference를 실행하지 않습니다.")
else:
    result = run_yolo11_video_inference(
        input_path=INPUT_PATH,
        out_dir=OUT_DIR,
        model_weights=MODEL_WEIGHTS,
        class_yaml=CLASS_YAML if CLASS_YAML.exists() else None,
        manual_classes=MANUAL_CLASSES or None,
        device=DEVICE,
        imgsz=IMGSZ,
        conf=CONF,
        iou=IOU,
        recursive=RECURSIVE,
        max_videos=MAX_VIDEOS,
        frame_stride=FRAME_STRIDE,
        start_seconds=START_SECONDS,
        end_seconds=END_SECONDS,
        max_frames=MAX_FRAMES,
        expected_class_count=EXPECTED_CLASS_COUNT,
        strict_class_count=STRICT_CLASS_COUNT,
        run_name=RUN_NAME,
        font_path=FONT_PATH,
        font_size=FONT_SIZE,
        line_width=LINE_WIDTH,
        overwrite=OVERWRITE,
        dry_run=DRY_RUN,
    )
    print(result.as_dict())
